In [2]:
import json

with open("reports/results_20260609_145957.json") as f:
    data = json.load(f)

print(type(data))

<class 'dict'>


In [3]:
print(data.keys())

dict_keys(['metadata', 'results', 'failures'])


In [4]:
import pandas as pd
df = pd.DataFrame(data["results"])

print(df.iloc[10]["metadata"])
df.iloc[0]

{'chain_name': 'fast_chain_sample_tket__qiskit_standard', 'num_steps': 2, 'steps': [{'name': 'tket', 'runner_type': 'tket'}, {'name': 'qiskit_standard', 'runner_type': 'qiskit_standard'}], 'step_durations': [0.9253990000579506, 0.01386212499346584], 'step_improvements': [-100.0, -91.07142857142857], 'total_duration_seconds': 0.9419590829638764}


circuit                                                      qft_8
runner                                                   qiskit_ai
optimizer                                                    chain
label                                              chain_qiskit_ai
metrics          {'depth': 31, 'two_qubit_gates': 52, 'two_qubi...
metadata         {'chain_name': 'fast_chain_sample_qiskit_ai', ...
artifact_path    brute_force_chains/cheap_optimizers/initial_te...
Name: 0, dtype: object

In [5]:
df["opt_level"] = df["metadata"].apply(lambda x: x.get("num_steps"))
df["opt_1"] = df["metadata"].apply(
    lambda x: x["steps"][0]["name"] if len(x["steps"]) > 0 else pd.NA
)
df["opt_2"] = df["metadata"].apply(
    lambda x: x["steps"][1]["name"] if len(x["steps"]) > 1 else pd.NA
)
df["opt_3"] = df["metadata"].apply(
    lambda x: x["steps"][2]["name"] if len(x["steps"]) > 2 else pd.NA
)
df["opt_chain"] = df.apply(
    lambda row: "__".join(
        [str(x) for x in [
            row["circuit"],
            row["opt_1"],
            row["opt_2"],
            row["opt_3"]
        ] if pd.notna(x)]
    ),
    axis=1
)
df["runtime"] = df["metadata"].apply(lambda x: x.get("total_duration_seconds"))
df["final_depth"] = df["metrics"].apply(lambda x: x.get("depth"))
df["final_two_qubit_gate_count"] = df["metrics"].apply(lambda x: x.get("two_qubit_gates"))
df["final_two_qubit_depth"] = df["metrics"].apply(lambda x: x.get("two_qubit_depth"))
df["final_total_gates"] = df["metrics"].apply(lambda x: x.get("total_gates"))
df.drop(columns=["metadata", "metrics", "runner", "label", ], inplace=True)
df.head()

,circuit,optimizer,artifact_path,opt_level,opt_1,opt_2,opt_3,opt_chain,runtime,final_depth,final_two_qubit_gate_count,final_two_qubit_depth,final_total_gates
0,qft_8,chain,brute_force_chains/cheap_optimizers/initial_te...,1,qiskit_ai,<NA>,<NA>,qft_8__qiskit_ai,1.002103,31,52,56,60
1,qft_8,chain,brute_force_chains/cheap_optimizers/initial_te...,1,qiskit_standard,<NA>,<NA>,qft_8__qiskit_standard,0.020029,112,108,64,281
2,qft_8,chain,brute_force_chains/cheap_optimizers/initial_te...,1,tket,<NA>,<NA>,qft_8__tket,1.145792,51,56,26,129
3,qft_8,chain,brute_force_chains/cheap_optimizers/initial_te...,2,qiskit_ai,qiskit_ai,<NA>,qft_8__qiskit_ai__qiskit_ai,0.418219,31,51,51,59
4,qft_8,chain,brute_force_chains/cheap_optimizers/initial_te...,2,qiskit_ai,qiskit_standard,<NA>,qft_8__qiskit_ai__qiskit_standard,0.178461,100,101,54,263


In [6]:
circuits = pd.read_csv("/Users/andrewweiland/UCCS_REU/quantum-optimizer-orchestration/dataset_analysis/circuits.csv")
circuits.head()

,Unnamed: 0,id,name,category,source,qasm_path,num_qubits,initial_depth,initial_two_qubit_gates,initial_two_qubit_depth,initial_total_gates,gate_density,two_qubit_ratio,created_at,optimizer_chain,original_circuit
0,0,1,efficient_su2_10_r2,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_10_...,10,17,18,11,78,7.800000,0.230769,2026-03-05 20:39:31,NaN,efficient_su2_10_r2
1,1,2,efficient_su2_12,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_12....,12,25,66,21,114,9.500000,0.578947,2026-03-05 20:39:31,NaN,efficient_su2_12
2,2,3,efficient_su2_12_r2,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_12_...,12,19,22,13,94,7.833333,0.234043,2026-03-05 20:39:31,NaN,efficient_su2_12_r2
3,3,4,efficient_su2_16,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_16....,16,33,120,29,184,11.500000,0.652174,2026-03-05 20:39:31,NaN,efficient_su2_16
4,4,5,efficient_su2_8,efficient_su2,local,benchmarks/ai_transpile/qasm/efficient_su2_8.qasm,8,27,56,21,104,13.000000,0.538462,2026-03-05 20:39:31,NaN,efficient_su2_8


In [7]:
df["parent_circuit_id"] = df["circuit"].map(
    circuits.set_index("name")["id"]
)
df.head()
df["parent_circuit"] = df["circuit"]
df["original_circuit_id"] = df["circuit"].map(
    circuits.set_index("name")["id"]
)
df["original_circuit"] = df["circuit"]
df["original_category"] = df["circuit"].map(
    circuits.set_index("name")["category"]
)
df["original_circuit_path"] = df["circuit"].map(
    circuits.set_index("name")["qasm_path"]
)

df["chain_name"] = df.apply(
    lambda row: "__".join(
        [str(x) for x in [
            row["opt_1"],
            row["opt_2"],
            row["opt_3"]
        ] if pd.notna(x)]
    ),
    axis=1
)

df["initial_num_qubits"] = df["circuit"].map(
    circuits.set_index("name")["num_qubits"]
)
df["initial_depth"] = df["circuit"].map(
    circuits.set_index("name")["initial_depth"]
)
df["initial_two_qubit_gates"] = df["circuit"].map(
    circuits.set_index("name")["initial_two_qubit_gates"]
)
df["initial_total_gates"] = df["circuit"].map(
    circuits.set_index("name")["initial_total_gates"]
)
df["final_num_qubits"] = df["initial_num_qubits"]

df = df[
    [
        "parent_circuit_id",
        "parent_circuit",
        "original_circuit_id",
        "original_circuit",
        "original_category",
        "original_circuit_path",
        "chain_name",
        "opt_level",
        "opt_1",
        "opt_2",
        "opt_3",
        "artifact_path",
        "initial_num_qubits",
        "initial_depth",
        "initial_two_qubit_gates",
        "initial_total_gates",
        "final_num_qubits",
        "final_depth",
        "final_two_qubit_gate_count",
        "final_total_gates",
        "runtime",
        "opt_chain"
    ]
]

df.head()

,parent_circuit_id,parent_circuit,original_circuit_id,original_circuit,original_category,original_circuit_path,chain_name,opt_level,opt_1,opt_2,...,initial_num_qubits,initial_depth,initial_two_qubit_gates,initial_total_gates,final_num_qubits,final_depth,final_two_qubit_gate_count,final_total_gates,runtime,opt_chain
0,65,qft_8,65,qft_8,qft,benchmarks/ai_transpile/qasm/qft_8.qasm,qiskit_ai,1,qiskit_ai,<NA>,...,8,15,28,36,8,31,52,60,1.002103,qft_8__qiskit_ai
1,65,qft_8,65,qft_8,qft,benchmarks/ai_transpile/qasm/qft_8.qasm,qiskit_standard,1,qiskit_standard,<NA>,...,8,15,28,36,8,112,108,281,0.020029,qft_8__qiskit_standard
2,65,qft_8,65,qft_8,qft,benchmarks/ai_transpile/qasm/qft_8.qasm,tket,1,tket,<NA>,...,8,15,28,36,8,51,56,129,1.145792,qft_8__tket
3,65,qft_8,65,qft_8,qft,benchmarks/ai_transpile/qasm/qft_8.qasm,qiskit_ai__qiskit_ai,2,qiskit_ai,qiskit_ai,...,8,15,28,36,8,31,51,59,0.418219,qft_8__qiskit_ai__qiskit_ai
4,65,qft_8,65,qft_8,qft,benchmarks/ai_transpile/qasm/qft_8.qasm,qiskit_ai__qiskit_standard,2,qiskit_ai,qiskit_standard,...,8,15,28,36,8,100,101,263,0.178461,qft_8__qiskit_ai__qiskit_standard


In [8]:
df.to_csv("hardware_aware_test_results.csv", index=False)